# 0.5. Data Setup

One-time setup — run this before notebooks 0–2.5 on any new device.

Downloads:
1. AHN grid shapefile (TU Delft GeoTiles)
2. BGT road layers via PDOK WFS (7 CSV files used by notebook 2)
3. BGT parkeervakken GeoJSON (used by CarFuser in notebook 2.5)

All paths are read from `config.py`. Run Cell 4 at the end to verify everything is in place.

In [ ]:
import io
import zipfile
import requests
import geopandas as gpd
import pandas as pd
from pathlib import Path

import sys
sys.path.insert(0, str(Path('.').resolve()))
from config import AHN_RAW_DIR, AHN_GRID_SHP, BGT_DIR, SETUP_TILECODES

## 1. AHN Grid Shapefile

Used by `0. Get AHN tiles.ipynb` to find which AHN tiles overlap a given pointcloud tile.

In [ ]:
ahn_shp_dir = AHN_GRID_SHP.parent
ahn_shp_dir.mkdir(parents=True, exist_ok=True)

if AHN_GRID_SHP.exists():
    print(f"AHN grid shapefile already present: {AHN_GRID_SHP}")
else:
    # Hosted as a release asset — original TU Delft GeoTiles URL is no longer stable
    url = "https://github.com/MinkeVerweij/UPSW_V2/releases/download/v1.0-data/AHN_subunits_GeoTiles.zip"
    print(f"Downloading AHN grid shapefile...")
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extractall(ahn_shp_dir)
    print(f"Extracted to {ahn_shp_dir}:")
    for f in sorted(ahn_shp_dir.iterdir()):
        print(f"  {f.name}")

## 2. BGT Road Layers (wegdeel) via PDOK WFS

Queries PDOK BGT WFS for `bgt:wegdeel` features within the A10 bounding box and splits them into the 7 CSV files expected by `2. Ground and Road fusion.ipynb`.

Format: semicolon-delimited CSV with columns `bgt-functie` and `geometrie` (WKT), matching the reader in `utils/bgt_fuser.py`.

In [ ]:
BGT_DIR.mkdir(parents=True, exist_ok=True)

FUNCTION_TO_LAYER = {
    "rijbaan lokale weg":     "BGT_WGL_rijbaan_lokale_weg",
    "rijbaan regionale weg":  "BGT_WGL_rijbaan_regionale_weg",
    "rijbaan autoweg":        "BGT_WGL_rijbaan_autoweg",
    "rijbaan autosnelweg":    "BGT_WGL_rijbaan_autosnelweg",
    "parkeervlak":            "BGT_WGL_parkeervlak",
    "OV-baan":                "BGT_WGL_ov-baan",
    "fietspad":               "BGT_WGL_fietspad",
}

already_present = all((BGT_DIR / f"{v}.csv").exists() for v in FUNCTION_TO_LAYER.values())
if already_present:
    print("All BGT road layer CSVs already present.")
else:
    CRS_RD = "http://www.opengis.net/def/crs/EPSG/0/28992"
    PAD = 200  # metres padding around each tile's lower-left corner

    # Query one tile at a time — each gives < 1000 features, no cursor pagination needed.
    # Tile bbox: (x0 - PAD, y0 - PAD, x0 + 700 + PAD, y0 + 700 + PAD) covers any
    # tile up to 700 m on a side, which is larger than any Cyclomedia tile in this project.
    print(f"Downloading BGT wegdeel for {len(SETUP_TILECODES)} tile(s): {SETUP_TILECODES}")
    all_features = []
    for tc in SETUP_TILECODES:
        x0, y0 = int(tc.split("_")[0]), int(tc.split("_")[1])
        bbox = f"{x0 - PAD},{y0 - PAD},{x0 + 700 + PAD},{y0 + 700 + PAD}"
        url = (
            "https://api.pdok.nl/lv/bgt/ogc/v1_0/collections/wegdeel/items"
            f"?limit=1000&bbox={bbox}&bbox-crs={CRS_RD}&crs={CRS_RD}"
        )
        print(f"  {tc}...", end=" ", flush=True)
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        data = r.json()
        feats = data.get("features", [])
        all_features.extend(feats)
        print(f"{len(feats)} features")
        # Safety: follow next link if a single tile somehow exceeds 1000 features
        next_url = next((l["href"] for l in data.get("links", []) if l.get("rel") == "next"), None)
        while next_url:
            r = requests.get(next_url, timeout=120)
            r.raise_for_status()
            data = r.json()
            feats = data.get("features", [])
            all_features.extend(feats)
            next_url = next((l["href"] for l in data.get("links", []) if l.get("rel") == "next"), None)

    gdf = gpd.GeoDataFrame.from_features(all_features, crs="EPSG:28992")
    gdf = gdf.drop_duplicates(subset=["lokaal_id"])
    print(f"Total after dedup: {len(gdf)} features")

    for func_val, layer_name in FUNCTION_TO_LAYER.items():
        subset = gdf[gdf["functie"] == func_val].copy()
        subset["geometrie"] = subset.geometry.to_wkt()
        out = subset.rename(columns={"functie": "bgt-functie"})[["bgt-functie", "geometrie"]]
        out.to_csv(BGT_DIR / f"{layer_name}.csv", sep=";", index=False)
        print(f"  {layer_name}.csv  ({len(out)} features)")

    print("Done.")

## 3. BGT Parkeervakken

Parking bay polygons used by `CarFuser` in `2.5. BGT Object Labeling.ipynb`.

In [ ]:
parkeer_out = BGT_DIR / "bgt_parkeervakken.json"

if parkeer_out.exists():
    print(f"bgt_parkeervakken.json already present ({parkeer_out})")
else:
    from config import A10_BBOX_RD
    xmin, ymin, xmax, ymax = A10_BBOX_RD
    CRS_RD = "http://www.opengis.net/def/crs/EPSG/0/28992"

    parkeervlak_csv = BGT_DIR / "BGT_WGL_parkeervlak.csv"
    if parkeervlak_csv.exists():
        from shapely import wkt as shapely_wkt
        from shapely.geometry import box as shapely_box
        a10_box = shapely_box(xmin, ymin, xmax, ymax)
        df_p = pd.read_csv(parkeervlak_csv, sep=";")
        df_p["geometry"] = df_p["geometrie"].apply(shapely_wkt.loads)
        parkeer = gpd.GeoDataFrame(df_p, geometry="geometry", crs="EPSG:28992")
        parkeer = parkeer[parkeer.geometry.intersects(a10_box)].copy()
        print(f"Loaded {len(parkeer)} parkeervlak features from CSV (clipped to A10 ring).")
    else:
        # Query PDOK in 1 km sub-tiles — avoids the 1000-feature page limit
        # and ensures full A10-ring coverage regardless of SETUP_TILECODES.
        STEP = 1000
        seen_ids = set()
        all_features = []
        xs = range(xmin, xmax, STEP)
        print(f"Querying PDOK BGT for {len(xs) * len(range(ymin, ymax, STEP))} sub-tiles...")
        for x0 in xs:
            for y0 in range(ymin, ymax, STEP):
                bbox = f"{x0},{y0},{min(x0+STEP,xmax)},{min(y0+STEP,ymax)}"
                url = (
                    "https://api.pdok.nl/lv/bgt/ogc/v1_0/collections/wegdeel/items"
                    f"?limit=1000&bbox={bbox}&bbox-crs={CRS_RD}&crs={CRS_RD}"
                )
                r = requests.get(url, timeout=120)
                r.raise_for_status()
                data = r.json()
                for feat in data.get("features", []):
                    fid = feat.get("id") or feat.get("properties", {}).get("lokaal_id")
                    if fid not in seen_ids:
                        seen_ids.add(fid)
                        all_features.append(feat)
        gdf_all = gpd.GeoDataFrame.from_features(all_features, crs="EPSG:28992")
        parkeer = gdf_all[gdf_all["functie"] == "parkeervlak"].copy()
        print(f"Found {len(parkeer)} unique parkeervlak features across A10 ring.")

    parkeer.to_file(str(parkeer_out), driver="GeoJSON")
    print(f"Saved {len(parkeer)} features -> {parkeer_out}")


## 4. Verification Checklist

In [ ]:
required = [
    (AHN_GRID_SHP,                            "AHN grid shapefile"),
    (BGT_DIR / "BGT_WGL_rijbaan_lokale_weg.csv",    "BGT rijbaan lokale weg"),
    (BGT_DIR / "BGT_WGL_rijbaan_regionale_weg.csv", "BGT rijbaan regionale weg"),
    (BGT_DIR / "BGT_WGL_rijbaan_autoweg.csv",       "BGT rijbaan autoweg"),
    (BGT_DIR / "BGT_WGL_rijbaan_autosnelweg.csv",   "BGT rijbaan autosnelweg"),
    (BGT_DIR / "BGT_WGL_parkeervlak.csv",           "BGT parkeervlak"),
    (BGT_DIR / "BGT_WGL_ov-baan.csv",               "BGT OV-baan"),
    (BGT_DIR / "BGT_WGL_fietspad.csv",              "BGT fietspad"),
    (BGT_DIR / "bgt_parkeervakken.json",            "BGT parkeervakken GeoJSON"),
]

all_ok = True
for path, label in required:
    ok = Path(path).exists()
    mark = "✓" if ok else "✗"
    print(f"  {mark}  {label}")
    if not ok:
        all_ok = False

print()
if all_ok:
    print("All required files present. Ready to run notebooks 0–2.5.")
else:
    print("Some files are missing — re-run the cells above.")